# Домашнє завдання 5 - CTE та Materialized View

## 1. Setup і PostgreSQL

Курс вимагає `pgserver`. Якщо поточний Colab runtime не підтримує пакет, використайте 2026.07.

In [ ]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub
import sys
import re
from pathlib import Path
import pandas as pd
import pgserver
import kagglehub
from sqlalchemy import create_engine, text
from IPython.display import display, Markdown

print('Python:', sys.version)
pg = pgserver.get_server('/tmp/hw5_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

def execute_script(sql_script):
    raw = engine.raw_connection()
    try:
        with raw.cursor() as cur:
            cur.execute(sql_script)
        raw.commit()
    finally:
        raw.close()

def execute_autocommit(sql_script):
    raw = engine.raw_connection()
    old = raw.autocommit
    try:
        raw.autocommit = True
        with raw.cursor() as cur:
            cur.execute(sql_script)
    finally:
        raw.autocommit = old
        raw.close()

def run_query(sql, params=None):
    return pd.read_sql(text(sql), engine, params=params or {})

def scalar(sql, params=None):
    with engine.connect() as conn:
        return conn.execute(text(sql), params or {}).scalar_one()

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


## 2. Отримання Olist

Використовується публічний Kaggle dataset `olistbr/brazilian-ecommerce`. Потрібні таблиці customers, orders, order_items, products і sellers.

In [ ]:
dataset_dir = Path(kagglehub.dataset_download('olistbr/brazilian-ecommerce'))
print('Kaggle dataset directory:', dataset_dir)

files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
}

resolved_files = {}
for table_name, file_name in files.items():
    matches = list(dataset_dir.rglob(file_name))
    if not matches:
        raise FileNotFoundError(f'Не знайдено {file_name} у {dataset_dir}')
    resolved_files[table_name] = matches[0]
    print(f'{table_name}: {matches[0]}')

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Kaggle dataset directory: /kaggle/input/brazilian-ecommerce
olist_customers_raw: /kaggle/input/brazilian-ecommerce/olist_customers_dataset.csv
olist_orders_raw: /kaggle/input/brazilian-ecommerce/olist_orders_dataset.csv
olist_order_items_raw: /kaggle/input/brazilian-ecommerce/olist_order_items_dataset.csv
olist_products_raw: /kaggle/input/brazilian-ecommerce/olist_products_dataset.csv
olist_sellers_raw: /kaggle/input/brazilian-ecommerce/olist_sellers_dataset.csv


## 3. Raw-рівень

Raw-таблиці відображають вихідні CSV і використовуються лише як проміжний шар перед typed-схемою.

In [ ]:
raw_shapes = []
for table_name, path in resolved_files.items():
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    raw_shapes.append((table_name, len(df), len(df.columns)))
    print(table_name, df.shape)

display(pd.DataFrame(raw_shapes, columns=['table_name','n_rows','n_columns']))

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


,table_name,n_rows,n_columns
0,olist_customers_raw,99441,5
1,olist_orders_raw,99441,8
2,olist_order_items_raw,112650,7
3,olist_products_raw,32951,9
4,olist_sellers_raw,3095,4


## 4. Typed-схема

У вихідному Olist CSV назви `product_name_lenght` та `product_description_lenght` містять помилку `lenght`. У typed-схемі використовуються нормалізовані назви з `length`.

In [ ]:
typed_ddl = r"""
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;
DROP TABLE IF EXISTS hw5_referrals CASCADE;
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders CASCADE;
DROP TABLE IF EXISTS olist_customers CASCADE;
DROP TABLE IF EXISTS olist_products CASCADE;
DROP TABLE IF EXISTS olist_sellers CASCADE;

CREATE TABLE olist_customers (
    customer_id TEXT PRIMARY KEY,
    customer_unique_id TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city TEXT,
    customer_state CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status TEXT NOT NULL,
    order_purchase_timestamp TIMESTAMP NOT NULL,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN ('created','approved','invoiced','processing','shipped','delivered','unavailable','canceled'))
);

CREATE TABLE olist_products (
    product_id TEXT PRIMARY KEY,
    product_category_name TEXT,
    product_name_length INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city TEXT,
    seller_state CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id INTEGER NOT NULL,
    product_id TEXT REFERENCES olist_products(product_id),
    seller_id TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price NUMERIC(12,2) CHECK (price IS NULL OR price >= 0),
    freight_value NUMERIC(12,2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);
"""
execute_script(typed_ddl)
print('Typed-схему створено.')

Typed-схему створено.


## 5. Raw -> typed через INSERT ... SELECT

In [ ]:
typed_load = r"""
INSERT INTO olist_customers
SELECT customer_id, customer_unique_id, customer_zip_code_prefix::INTEGER, customer_city, customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT order_id, customer_id, order_status,
       order_purchase_timestamp::TIMESTAMP,
       order_approved_at::TIMESTAMP,
       order_delivered_carrier_date::TIMESTAMP,
       order_delivered_customer_date::TIMESTAMP,
       order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT product_id, product_category_name,
       product_name_lenght::INTEGER,
       product_description_lenght::INTEGER,
       product_photos_qty::INTEGER,
       product_weight_g::INTEGER,
       product_length_cm::INTEGER,
       product_height_cm::INTEGER,
       product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT seller_id, seller_zip_code_prefix::INTEGER, seller_city, seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT order_id, order_item_id::INTEGER, product_id, seller_id,
       shipping_limit_date::TIMESTAMP,
       price::NUMERIC(12,2),
       freight_value::NUMERIC(12,2)
FROM olist_order_items_raw;
"""
execute_script(typed_load)
print('Raw -> typed завершено.')

Raw -> typed завершено.


### 5.1 Smoke-test row counts

In [ ]:
smoke = run_query("""
SELECT 'customers' AS table_name, (SELECT COUNT(*) FROM olist_customers_raw) AS raw_rows, (SELECT COUNT(*) FROM olist_customers) AS typed_rows
UNION ALL SELECT 'orders', (SELECT COUNT(*) FROM olist_orders_raw), (SELECT COUNT(*) FROM olist_orders)
UNION ALL SELECT 'order_items', (SELECT COUNT(*) FROM olist_order_items_raw), (SELECT COUNT(*) FROM olist_order_items)
UNION ALL SELECT 'products', (SELECT COUNT(*) FROM olist_products_raw), (SELECT COUNT(*) FROM olist_products)
UNION ALL SELECT 'sellers', (SELECT COUNT(*) FROM olist_sellers_raw), (SELECT COUNT(*) FROM olist_sellers)
ORDER BY table_name;
""")
display(smoke)
all_equal = bool((smoke['raw_rows'] == smoke['typed_rows']).all())
display(Markdown(f'**Інтерпретація:** кількість рядків raw і typed збігається для всіх таблиць: **{"так" if all_equal else "ні"}**.'))

,table_name,raw_rows,typed_rows
0,customers,99441,99441
1,order_items,112650,112650
2,orders,99441,99441
3,products,32951,32951
4,sellers,3095,3095


**Інтерпретація:** кількість рядків raw і typed збігається для всіх таблиць: **так**.

# 6. CTE-chain feature engineering pipeline

Pipeline містить 7 named stages плюс `reference_date`. Обов'язкові features: RFM, `cohort_quarter`, rolling AOV 30d, `cancellation_rate`, `preferred_category`.

In [ ]:
feature_pipeline_sql = r"""
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT o.customer_id, o.order_id, o.order_status,
           o.order_purchase_timestamp AS order_ts,
           o.order_purchase_timestamp::DATE AS order_date,
           COALESCE(SUM(COALESCE(oi.price,0) + COALESCE(oi.freight_value,0)),0)::NUMERIC(12,2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.customer_id, o.order_id, o.order_status, o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT s.customer_id,
           (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
           COUNT(*)::INTEGER AS frequency,
           SUM(s.order_total)::NUMERIC(12,2) AS monetary,
           MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT r.*, DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT s.customer_id, s.order_id, s.order_ts, s.order_total,
           AVG(s.order_total) OVER (
               PARTITION BY s.customer_id
               ORDER BY s.order_ts
               RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
           )::NUMERIC(12,2) AS avg_order_value_30d,
           ROW_NUMBER() OVER (
               PARTITION BY s.customer_id
               ORDER BY s.order_ts DESC, s.order_id DESC
           ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT customer_id,
           AVG(CASE WHEN order_status='canceled' THEN 1.0 ELSE 0.0 END)::NUMERIC(6,4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT o.customer_id,
           COALESCE(p.product_category_name,'(unknown)') AS product_category_name,
           SUM(COALESCE(oi.price,0)+COALESCE(oi.freight_value,0))::NUMERIC(12,2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name,'(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT customer_id, product_category_name, category_revenue
    FROM (
        SELECT cr.*,
               ROW_NUMBER() OVER (
                   PARTITION BY cr.customer_id
                   ORDER BY cr.category_revenue DESC, cr.product_category_name
               ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT c.customer_id,
       c.customer_state,
       COALESCE(coh.recency_days,9999) AS recency_days,
       COALESCE(coh.frequency,0) AS frequency,
       COALESCE(coh.monetary,0)::NUMERIC(12,2) AS monetary,
       coh.first_order_date,
       coh.cohort_quarter,
       COALESCE(aov.avg_order_value_30d,0)::NUMERIC(12,2) AS avg_order_value_30d_recent,
       COALESCE(cancel.cancellation_rate,0)::NUMERIC(6,4) AS cancellation_rate,
       COALESCE(pc.product_category_name,'(none)') AS preferred_category,
       COALESCE(pc.category_revenue,0)::NUMERIC(12,2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov ON aov.customer_id = c.customer_id AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc ON pc.customer_id = c.customer_id
"""

In [ ]:
feature_sample = run_query(
    feature_pipeline_sql
    + "\nORDER BY monetary DESC NULLS LAST, customer_id\nLIMIT 20"
)
display(feature_sample)
display(Markdown(
    f"**Інтерпретація:** preview містить **{len(feature_sample)}** customer-level rows. "
    "Кожен CTE stage має одну відповідальність і може перевірятися окремо."
))

,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0,artes,6726.66
5,05455dfa7cd02f13d132aa7a6a9729c6,MG,328,1,6081.54,2017-11-24,2017-10-01,6081.54,0.0,agro_industria_e_comercio,6081.54
6,df55c14d1476a9a3467f131269c2477f,RJ,565,1,4950.34,2017-04-01,2017-04-01,4950.34,0.0,eletroportateis,4950.34
7,e0a2412720e9ea4f26c1ac985f6a7358,GO,98,1,4809.44,2018-07-12,2018-07-01,4809.44,1.0,cool_stuff,4809.44
8,24bbf5fd2f2e1b359ee7de94defc4a15,SP,548,1,4764.34,2017-04-18,2017-04-01,4764.34,0.0,eletroportateis,4764.34
9,3d979689f636322c62418b6346b1c6d2,PB,118,1,4681.78,2018-06-22,2018-04-01,4681.78,0.0,pcs,4681.78


**Інтерпретація:** preview містить **20** customer-level rows. Кожен CTE stage має одну відповідальність і може перевірятися окремо.

## 6.1 Перевірка проміжного stage

In [ ]:
stage2_preview = run_query("""
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date FROM olist_orders
),
stage1_orders_total AS (
    SELECT o.customer_id, o.order_id, o.order_purchase_timestamp::DATE AS order_date,
           COALESCE(SUM(COALESCE(oi.price,0)+COALESCE(oi.freight_value,0)),0)::NUMERIC(12,2) AS order_total
    FROM olist_orders o
    LEFT JOIN olist_order_items oi ON oi.order_id=o.order_id
    GROUP BY o.customer_id,o.order_id,o.order_purchase_timestamp
),
stage2_rfm AS (
    SELECT s.customer_id,
           (rd.as_of_date-MAX(s.order_date))::INTEGER AS recency_days,
           COUNT(*)::INTEGER AS frequency,
           SUM(s.order_total)::NUMERIC(12,2) AS monetary,
           MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total s
    CROSS JOIN reference_date rd
    GROUP BY s.customer_id,rd.as_of_date
)
SELECT * FROM stage2_rfm ORDER BY monetary DESC LIMIT 10;
""")
display(stage2_preview)

,customer_id,recency_days,frequency,monetary,first_order_date
0,1617b1357756262bfa56ab541c47bc16,384,1,13664.08,2017-09-29
1,ec5b2ba62e574342386871631fafd3fc,95,1,7274.88,2018-07-15
2,c6e2731c5b391845f6800c97401a43a9,613,1,6929.31,2017-02-12
3,f48d464a0baaea338cb25f816991ab1f,85,1,6922.21,2018-07-25
4,3fd6777bbce08a352fddd04e4a7cc8f6,512,1,6726.66,2017-05-24
5,05455dfa7cd02f13d132aa7a6a9729c6,328,1,6081.54,2017-11-24
6,df55c14d1476a9a3467f131269c2477f,565,1,4950.34,2017-04-01
7,e0a2412720e9ea4f26c1ac985f6a7358,98,1,4809.44,2018-07-12
8,24bbf5fd2f2e1b359ee7de94defc4a15,548,1,4764.34,2017-04-18
9,3d979689f636322c62418b6346b1c6d2,118,1,4681.78,2018-06-22


# 7. Recursive CTE для referral chain

Olist не містить referral graph, тому створюється синтетична `hw5_referrals`. Anchor використовує `NOT EXISTS`, а recursion має `path_ids` і depth guard.

In [ ]:
referral_setup = r"""
DROP TABLE IF EXISTS hw5_referrals CASCADE;
CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

WITH ordered_customers AS (
    SELECT customer_id, ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 10
)
INSERT INTO hw5_referrals (referrer_customer_id,referred_customer_id)
SELECT prev.customer_id,nxt.customer_id
FROM ordered_customers prev
JOIN ordered_customers nxt ON nxt.rn=prev.rn+1
ON CONFLICT (referred_customer_id) DO NOTHING;
"""
execute_script(referral_setup)
display(run_query('SELECT * FROM hw5_referrals ORDER BY referrer_customer_id,referred_customer_id'))

,referrer_customer_id,referred_customer_id,referred_at
0,00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140,2026-10-18 11:23:04.673016+00:00
1,000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079,2026-10-18 11:23:04.673016+00:00
2,0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5,2026-10-18 11:23:04.673016+00:00
3,0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb,2026-10-18 11:23:04.673016+00:00
4,000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652,2026-10-18 11:23:04.673016+00:00
5,0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086,2026-10-18 11:23:04.673016+00:00
6,000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282,2026-10-18 11:23:04.673016+00:00
7,00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c,2026-10-18 11:23:04.673016+00:00
8,00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130,2026-10-18 11:23:04.673016+00:00


In [ ]:
recursive_sql = r"""
WITH RECURSIVE referral_depth AS (
    -- ANCHOR: roots, які не були referred
    SELECT c.customer_id,
           1 AS depth,
           ARRAY[c.customer_id]::TEXT[] AS path_ids,
           c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1 FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- RECURSIVE: перейти до referred customers
    SELECT r.referred_customer_id,
           rd.depth + 1,
           rd.path_ids || r.referred_customer_id,
           rd.path_text || ' -> ' || r.referred_customer_id
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd ON rd.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY(rd.path_ids)
      AND rd.depth < 50
)
SELECT depth, COUNT(*) AS n_customers, MAX(path_text) AS sample_path
FROM referral_depth
GROUP BY depth
ORDER BY depth;
"""
recursive_result = run_query(recursive_sql)
display(recursive_result)

,depth,n_customers,sample_path
0,1,99432,ffffe8b65bbe3087b653a978c870db99
1,2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
2,3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
3,4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
4,5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
5,6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
6,7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
7,8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
8,9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
9,10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...


При циклі `A -> B -> A` другий перехід у `A` блокується перевіркою `NOT next_id = ANY(path_ids)`. Додатково `depth < 50` захищає від runaway recursion у разі пошкоджених даних.

# 8. Materialized View

MV містить повний результат feature pipeline. `UNIQUE INDEX` на `customer_id` потрібний для `REFRESH ... CONCURRENTLY`.

In [ ]:
execute_script('DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;')

create_mv_sql = (
    'CREATE MATERIALIZED VIEW mv_olist_customer_features AS\n'
    + feature_pipeline_sql
    + ';'
)
execute_script(create_mv_sql)

execute_script("""
CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
ON mv_olist_customer_features (customer_id);
""")
print('MV та UNIQUE INDEX створено.')

MV та UNIQUE INDEX створено.


In [ ]:
display(run_query('SELECT COUNT(*) AS n_rows FROM mv_olist_customer_features'))
display(run_query('SELECT * FROM mv_olist_customer_features ORDER BY monetary DESC NULLS LAST, customer_id LIMIT 5'))

,n_rows
0,99441


,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0,artes,6726.66


## 8.1 Concurrent refresh

In [ ]:
execute_autocommit('REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;')
print('REFRESH MATERIALIZED VIEW CONCURRENTLY виконано успішно.')

REFRESH MATERIALIZED VIEW CONCURRENTLY виконано успішно.


# 9. EXPLAIN (ANALYZE, BUFFERS)

Порівнюється повне переобчислення feature pipeline і lookup однієї сутності з MV.

## 9.1 Full pipeline

In [ ]:
explain_full = run_query(
    'EXPLAIN (ANALYZE, BUFFERS)\n'
    + feature_pipeline_sql
    + '\nORDER BY monetary DESC NULLS LAST, customer_id\nLIMIT 10'
)
display(explain_full)

,QUERY PLAN
0,Limit (cost=97944.75..97944.78 rows=10 width=...
1,"Buffers: shared hit=12448, temp read=9962 wr..."
2,CTE stage1_orders_total
3,-> Finalize GroupAggregate (cost=16725.6...
4,Group Key: o_1.order_id
...,...
143,...
144,Planning:
145,Buffers: shared hit=3
146,Planning Time: 1.087 ms


## 9.2 MV lookup

In [ ]:
explain_mv = run_query("""
EXPLAIN (ANALYZE, BUFFERS)
WITH target_customer AS (
    SELECT customer_id FROM olist_customers ORDER BY customer_id LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t ON t.customer_id = mv.customer_id;
""")
display(explain_mv)

,QUERY PLAN
0,Nested Loop (cost=0.83..8.98 rows=1 width=154...
1,Buffers: shared hit=5 read=3
2,-> Limit (cost=0.42..0.55 rows=1 width=32)...
3,Buffers: shared hit=4
4,-> Index Only Scan using olist_custom...
5,Heap Fetches: 1
6,Buffers: shared hit=4
7,-> Index Scan using idx_mv_olist_customer_f...
8,Index Cond: (customer_id = olist_custo...
9,Buffers: shared hit=1 read=3


## 9.3 Автоматичне резюме планів

In [ ]:
def explain_text(df):
    return '\n'.join(df.iloc[:, 0].astype(str).tolist())

def extract_execution_time(plan_text):
    m = re.search(r'Execution Time:\s*([0-9.]+)\s*ms', plan_text)
    return float(m.group(1)) if m else None

def detect_nodes(plan_text):
    nodes = [
        'Index Scan', 'Index Only Scan', 'Seq Scan',
        'Hash Join', 'Merge Join', 'Nested Loop',
        'HashAggregate', 'GroupAggregate', 'WindowAgg', 'Sort'
    ]
    return [n for n in nodes if n in plan_text]

full_plan_text = explain_text(explain_full)
mv_plan_text = explain_text(explain_mv)
full_ms = extract_execution_time(full_plan_text)
mv_ms = extract_execution_time(mv_plan_text)
full_nodes = detect_nodes(full_plan_text)
mv_nodes = detect_nodes(mv_plan_text)

display(pd.DataFrame([
    {
        'scenario': 'full_pipeline',
        'execution_ms': full_ms,
        'detected_nodes': ', '.join(full_nodes),
    },
    {
        'scenario': 'mv_lookup',
        'execution_ms': mv_ms,
        'detected_nodes': ', '.join(mv_nodes),
    },
]))

,scenario,execution_ms,detected_nodes
0,full_pipeline,2679.338,"Seq Scan, Hash Join, Nested Loop, HashAggregat..."
1,mv_lookup,0.112,"Index Scan, Index Only Scan, Nested Loop"


# 10. Reflection

Reflection формується з фактичними execution times і plan nodes поточного запуску.

In [ ]:
full_ms_text = f'{full_ms:.3f} ms' if full_ms is not None else 'не вдалося витягти автоматично'
mv_ms_text = f'{mv_ms:.3f} ms' if mv_ms is not None else 'не вдалося витягти автоматично'
full_nodes_text = ', '.join(full_nodes) if full_nodes else 'оператори не розпізнані автоматично'
mv_nodes_text = ', '.join(mv_nodes) if mv_nodes else 'оператори не розпізнані автоматично'

reflection = f"""
### Reflection

У моєму запуску `EXPLAIN (ANALYZE, BUFFERS)` для повного feature pipeline показав execution time **{full_ms_text}**, а lookup із Materialized View - **{mv_ms_text}**. У full-plan були помітні оператори **{full_nodes_text}**, тоді як для MV lookup - **{mv_nodes_text}**. Це демонструє різницю між повторним обчисленням JOIN, GROUP BY та window functions і читанням уже підготовленого customer-level feature row.

Найдорожчою частиною pipeline я вважаю побудову `preferred_category` разом із category revenue та rolling AOV. Вони потребують JOIN з `order_items` і `products`, агрегації, сортування та `WindowAgg`. На великих даних саме такі операції масштабуються гірше за простий key lookup. Тому кешування фінального feature set через Materialized View є виправданим для read-heavy ML-serving.

Для e-commerce scoring я б обрав refresh щогодини, якщо модель використовується оперативно, або щодоби для batch-моделей, де допустима більша staleness. Звичайний `VIEW` був би кращим, якщо запит дешевий, виконується рідко або бізнес вимагає бачити зміни базових таблиць одразу.

Recursive CTE завершується природним вичерпанням зв'язків, перевіркою відвіданих `path_ids` і guard-умовою `depth < 50`. При циклі `A -> B -> A` повторний вузол не додається, тому нескінченної рекурсії не буде.

Головний trade-off у цьому pipeline - **freshness проти latency та recompute-cost**. MV зменшує latency читання і переносить дорогі обчислення в контрольований refresh-cycle, але потребує storage і допускає staleness між refresh-операціями.
"""
display(Markdown(reflection))


### Reflection

У моєму запуску `EXPLAIN (ANALYZE, BUFFERS)` для повного feature pipeline показав execution time **2679.338 ms**, а lookup із Materialized View - **0.112 ms**. У full-plan були помітні оператори **Seq Scan, Hash Join, Nested Loop, HashAggregate, GroupAggregate, WindowAgg, Sort**, тоді як для MV lookup - **Index Scan, Index Only Scan, Nested Loop**. Це демонструє різницю між повторним обчисленням JOIN, GROUP BY та window functions і читанням уже підготовленого customer-level feature row.

Найдорожчою частиною pipeline я вважаю побудову `preferred_category` разом із category revenue та rolling AOV. Вони потребують JOIN з `order_items` і `products`, агрегації, сортування та `WindowAgg`. На великих даних саме такі операції масштабуються гірше за простий key lookup. Тому кешування фінального feature set через Materialized View є виправданим для read-heavy ML-serving.

Для e-commerce scoring я б обрав refresh щогодини, якщо модель використовується оперативно, або щодоби для batch-моделей, де допустима більша staleness. Звичайний `VIEW` був би кращим, якщо запит дешевий, виконується рідко або бізнес вимагає бачити зміни базових таблиць одразу.

Recursive CTE завершується природним вичерпанням зв'язків, перевіркою відвіданих `path_ids` і guard-умовою `depth < 50`. При циклі `A -> B -> A` повторний вузол не додається, тому нескінченної рекурсії не буде.

Головний trade-off у цьому pipeline - **freshness проти latency та recompute-cost**. MV зменшує latency читання і переносить дорогі обчислення в контрольований refresh-cycle, але потребує storage і допускає staleness між refresh-операціями.
